# BASIC RAG PIPELINE FROM SCRATCH

- here we will build a basic RAG pipeline
- we will use 14 blogs as dataset.
- these blogs talks about the LLM, AI, trends etc 
- we will use gemini-2.5 Flash LLM because its ideal for webapps and chatbots . It offers low latency and works on better TPU(Tensor Processing UNITS)

In [47]:
import os
import csv
import pandas as pd 
from tqdm.notebook import tqdm
import numpy as np
from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv #to load the env vars securily
from openai import OpenAI #to create the openai client in order to communicate with the openai endpoints



In [48]:
# Load variables from .env file
load_dotenv()

True

In [49]:
# Access the keys
openai_key = os.getenv("OPENAI_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")

print("Keys loaded successfully")

Keys loaded successfully


In [4]:
#here we have the choice whetehr to create the embeddings for all the chunked data or to use the precomputed embeddings
#if the flag is - 
# False: Generate the embedding for the dataset. (Associated cost with using OpenAI endpoint)
# True: Load the dataset that already has the embedding vectors.
load_embedding = False


In [ ]:
#download the articles 
#these articles we have taken from towards ai blog
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv

--2026-07-15 21:13:54--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 173646 (170K) [text/plain]
Saving to: ‘mini-llama-articles.csv’

mini-llama-articles 100%[===================>] 169.58K  --.-KB/s    in 0.08s   

2026-07-15 21:13:54 (1.96 MB/s) - ‘mini-llama-articles.csv’ saved [173646/173646]

--2026-07-15 21:13:55--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP r

### CHUNKING DATA 

- although there are many of chunking, for our iteration we are using a simple chunking methodology.
- we are chunking the text after every 1024 chars 

In [7]:
def split_into_chunks(text, chunk_size=1024):
    chunks = []
    for i in range(0,len(text),chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

In [ ]:
chunks = []

# Load the file as a CSV
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue #here we are skipping the 1st row because the first row is the header row and it does not provide any info
        chunks.extend(split_into_chunks(row[1]))
#if we have used append here then we would get list of list so that is why we used extend here so that we get a clean list 
#yes there is performance tradeoff as extend operation is O(n) while append is O(1)

In [14]:
print("number of articles:", idx)
print("number of chunks:", len(chunks))

number of articles: 14
number of chunks: 174


In [15]:
#lets get a proper dataframe 
df = pd.DataFrame(chunks, columns=["chunk"])



In [16]:
df.shape

(174, 1)

In [17]:
df.head()

,chunk
0,LLM Variants and Meta's Open Source Before she...
1,ational code model;Codel Llama - Python specia...
2,"erm ""multimodal"" refers to their ability to pr..."
3,"es it matter? LLM connections, like the LlamaI..."
4,understand data in the AI-driven future. Fro...


In [18]:
df.keys()

Index(['chunk'], dtype='object')

### GENERATE EMBEDDINGS 

In [ ]:
#this is our openai client, It will communicate the openai LLM open endpoint 
client = OpenAI()


In [21]:
#here we are using openai text-embedding-3-small model to create the embeddings
def get_embedding(text):
    try:
        text = text.replace("\n"," ") #replacing the newline
        res = client.embeddings.create(input=[text],model="text-embedding-3-small")
        return res.data[0].embedding
    except:
        return None


In [23]:
#embedding generation in action 
#this is the flag we have mentioned above
#if load_embedding == False then we will get the embeddings 
#if load_embedding == True then we will use the pre-existing/computed embeddings
if not load_embedding:
    print("generating embeddings...")
    embeddings = []
    for index,row in tqdm(df.iterrows()):
        embeddings.append(get_embedding(row["chunk"]))
    embedding_values = pd.Series(embeddings)
    df.insert(loc=1, column="embedding",value=embedding_values)
else:
    print("loaded the embedding files")
    df= pd.read_csv("mini-llama-article-with-embeddings.csv")
    df["embedding"] = df["embedding"].apply(lambda x: np.array(eval(x)),0)



generating embeddings...


0it [00:00, ?it/s]

### USER QUESTION 


In [24]:
# Define the user question, and convert it to embedding.
QUESTION = "How many parameters LLaMA2 model has?"
QUESTION_emb = get_embedding(QUESTION)

len(QUESTION_emb)

1536

### KLEIN TEST FÜR COSINE SIMILARIIES

In [25]:
BAD_SOURCE_emb = get_embedding("The sky is blue.")
GOOD_SOURCE_emb = get_embedding("LLaMA2 model has a total of 2B parameters.")

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

# A sample that how a good piece of text can achieve high similarity score compared
# to a completely unrelated text.
print("> Bad Response Score:", cosine_similarity([QUESTION_emb], [BAD_SOURCE_emb]))
print("> Good Response Score:", cosine_similarity([QUESTION_emb], [GOOD_SOURCE_emb]))

> Bad Response Score: [[0.02583581]]
> Good Response Score: [[0.83190382]]


/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [28]:
#now we will calculate the cosine similarity between the use question and each of the chunk embedding we have in our knowledge base 
cosine_similarities = cosine_similarity([QUESTION_emb], df["embedding"].tolist())

print(cosine_similarities)

[[0.46765602 0.46912976 0.25987883 0.29370217 0.31963703 0.40164534
  0.41474759 0.45248701 0.45940695 0.12606733 0.11751525 0.01345546
  0.22608057 0.21451279 0.10150473 0.33115032 0.10747553 0.34685975
  0.16309879 0.08738607 0.34816851 0.22847438 0.19203652 0.26472234
  0.24955085 0.34844254 0.24838201 0.32775136 0.41325995 0.41350428
  0.45752192 0.38312563 0.46913931 0.35643024 0.35401014 0.30148176
  0.29943405 0.29263368 0.40024878 0.4646871  0.39490304 0.41049787
  0.44693871 0.43181967 0.35911692 0.33973119 0.51354911 0.20936738
  0.40207215 0.32832147 0.428693   0.48175342 0.45035729 0.34253262
  0.32080646 0.42599045 0.24691352 0.18068208 0.23651507 0.34266439
  0.34379271 0.2048215  0.19769979 0.22442907 0.21108608 0.42306595
  0.26411588 0.30433071 0.33614761 0.38309262 0.23538305 0.24352995
  0.37078515 0.28030015 0.4905238  0.53046781 0.37840242 0.43812034
  0.37755103 0.39279185 0.30065552 0.41686787 0.46731445 0.45428895
  0.3516303  0.21229325 0.42614361 0.31607037 0.

/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [29]:

number_of_chunks_to_retrieve = 3

# Sort the scores
highest_index = np.argmax(cosine_similarities)

# Pick the N highest scored chunks
indices = np.argsort(cosine_similarities[0])[::-1][:number_of_chunks_to_retrieve]
print(indices)

[114  75  89]


In [31]:
# Look at the highest scored retrieved pieces of text
for idx, item in enumerate(df.chunk[indices]):
    print(f"> Chunk {idx+1}")
    print(item)
    print("----")

> Chunk 1
by Meta that ventures into both the AI and academic spaces. The model aims to help researchers, scientists, and engineers advance their work in exploring AI applications. It will be released under a non-commercial license to prevent misuse, and access will be granted to academic researchers, individuals, and organizations affiliated with the government, civil society, academia, and industry research facilities on a selective case-by-case basis. The sharing of codes and weights allows other researchers to test new approaches in LLMs. The LLaMA models have a range of 7 billion to 65 billion parameters. LLaMA-65B can be compared to DeepMind's Chinchilla and Google's PaLM. Publicly available unlabeled data was used to train these models, and training smaller foundational models require less computing power and resources. LLaMA 65B and 33B have been trained on 1.4 trillion tokens in 20 different languages, and according to the Facebook Artificial Intelligence Research (FAIR) team,

### PROMPT AUGMENTATION 

In [50]:
# Initialize client
client = genai.Client(api_key=google_key)

In [ ]:
# Use the Gemini API to answer the questions based on the retrieved pieces of text.
try:
    # System instructions for the AI assistant
    system_instruction = (
        "You are an assistant and expert in answering questions from a chunks of content. "
        "Only answer AI-related question, else say that you cannot answer this question."
    )

    # Create a user prompt with the user's question
    prompt = (
        "Read the following informations that might contain the context you require to answer the question. You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
        "Please provide an informative and accurate answer to the following question based on the available context. Be concise and take your time. \nQuestion: {}\nAnswer:"
    )

    # Add the retrieved pieces of text to the prompt
    formatted_prompt = prompt.format("".join(df.chunk[indices]), QUESTION)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=formatted_prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0),
            system_instruction=system_instruction,
            temperature=0.0,
        )
    )

    res = response.text

except Exception as e:
    print(f"An error occurred: {e}")

#print(res)

An error occurred: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}
